# 1️⃣ Setup and Load Data

## What This Section Does
In this first step, we initialize **Apache Spark** and load the credit card fraud dataset into a Spark DataFrame.

## Why This Matters
- Spark lets us work efficiently with large datasets
- The dataset is highly imbalanced, so we need a tool that can handle many rows quickly
- Loading the data into a DataFrame gives us a clean structure for filtering, counting, and sampling

## Code Goal
The code below:
- Starts a Spark session
- Reads `../data/creditcard.csv`
- Prints the total number of rows in the original dataset

## What to Notice
- The dataset has **284,807 rows** in total
- Only a very small portion of those rows are fraud cases, which is why balancing is needed later

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Initialize Spark
spark = SparkSession.builder \
    .appName("Fraud_Balancing") \
    .getOrCreate()

# Load the raw dataset
df = spark.read.csv("../data/creditcard.csv", header=True, inferSchema=True)

print(f"Original Dataset Size: {df.count()} rows")

your 131072x1 screen size is bogus. expect trouble
26/04/29 15:19:30 WARN Utils: Your hostname, SerbansLaptop resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/29 15:19:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 15:19:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/29 15:19:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Original Dataset Size: 284807 rows


# 2️⃣ Separate the Classes and Calculate the Sampling Fraction

## What This Section Does
Here we split the dataset into two groups:
- **Fraud transactions** where `Class = 1`
- **Normal transactions** where `Class = 0`

Then we count each group and calculate how much of the normal data we need to keep.

## Why This Matters
The dataset is extremely imbalanced, so if we train a model on the raw data, it may learn to predict only the normal class.

## Code Goal
The code below:
- Filters the original DataFrame into fraud and normal subsets
- Counts both subsets
- Computes `fraction_to_keep = fraud_count / normal_count`

## What This Fraction Means
If the fraud class has 492 rows and the normal class has 284,315 rows, then the fraction will be very small. That means we only keep a tiny random sample of the normal transactions so the dataset becomes more balanced.

In [2]:
# 1. Split the data into Fraud (Class 1) and Normal (Class 0)
fraud_df = df.filter(col("Class") == 1)
normal_df = df.filter(col("Class") == 0)

# 2. Count both datasets
fraud_count = fraud_df.count()
normal_count = normal_df.count()

print(f"Fraud cases: {fraud_count}")
print(f"Normal cases: {normal_count}")

# 3. Calculate the fraction needed to undersample the normal data
# We want the normal count to equal the fraud count
fraction_to_keep = fraud_count / normal_count
print(f"Fraction of normal data to keep: {fraction_to_keep:.5f} (roughly {fraction_to_keep * 100:.2f}%)")

Fraud cases: 492
Normal cases: 284315
Fraction of normal data to keep: 0.00173 (roughly 0.17%)


# 3️⃣ Perform the Undersampling

## What This Section Does
Now we use Spark to randomly sample the normal transactions and keep only a small percentage of them.

## Why This Matters
Undersampling helps us reduce the class imbalance so the fraud class becomes easier for a model to learn.

## Code Goal
The code below:
- Randomly samples the normal class using `sample()`
- Keeps only the fraction computed earlier
- Combines the fraud rows with the sampled normal rows using `unionByName()`
- Shuffles the final dataset so fraud rows do not appear all together

## Important Details
- `withReplacement=False` means the same row cannot be selected twice
- `seed=42` makes the sampling reproducible
- `orderBy(rand(seed=42))` randomizes the row order after merging the two groups

In [3]:
# 1. Randomly sample the Normal data
# withReplacement=False means we don't pick the same row twice
# seed=42 ensures we get the exact same random rows every time we run this code
undersampled_normal_df = normal_df.sample(withReplacement=False, fraction=fraction_to_keep, seed=42)

# 2. Combine the Fraud data with our new, smaller Normal data
balanced_df = fraud_df.unionByName(undersampled_normal_df)

# 3. Shuffle the combined dataset so the model doesn't read all frauds first
# We do this by ordering by a random number
from pyspark.sql.functions import rand
balanced_df = balanced_df.orderBy(rand(seed=42))

print(f"New Balanced Dataset Size: {balanced_df.count()} rows")

New Balanced Dataset Size: 991 rows


# 4️⃣ Verify the New Distribution

## What This Section Does
After undersampling, we check whether the dataset is now balanced.

## Why This Matters
We need to confirm that the new dataset contains roughly equal numbers of fraud and normal transactions before using it for training.

## Code Goal
The code below groups the balanced dataset by `Class` and counts how many rows are in each class.

## What You Should Expect
The output should show:
- A much smaller normal class
- The fraud class unchanged
- A dataset that is now far more balanced than the original

In [4]:
print("--- NEW BALANCED CLASS DISTRIBUTION ---")
balanced_df.groupBy("Class").count().show()

--- NEW BALANCED CLASS DISTRIBUTION ---
+-----+-----+
|Class|count|
+-----+-----+
|    1|  492|
|    0|  499|
+-----+-----+



# 5️⃣ Save to Parquet

## What This Section Does
Finally, we save the balanced dataset to disk in **Parquet** format.

## Why Parquet?
- It is much faster than CSV for Spark and other big-data tools
- It stores data in a compact binary format
- It preserves schema information and improves read performance

## Code Goal
The code below:
- Writes the balanced dataset to `../data/processed/balanced_creditcard.parquet`
- Uses `overwrite` mode so the file is replaced if it already exists
- Confirms that the balanced data was saved successfully

## Why This Is Useful
Saving the processed data means we do not need to repeat the undersampling step every time we start a new analysis or model training session.

In [5]:
output_path = "../data/processed/balanced_creditcard.parquet"

# Write the data to the processed folder
balanced_df.write.mode("overwrite").parquet(output_path)

print(f"Balanced data successfully saved to: {output_path}")

26/04/29 15:21:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Balanced data successfully saved to: ../data/processed/balanced_creditcard.parquet
